# `tbl-results-mse`: OLS treatment-effect error and empirical cross-entropy by dataset and regime

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "tbl-results-mse"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/tbl-results-mse.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [3]:
MODEL_LABELS = {"kev-0.8b": "KEV-0.8B", "kev-4b": "KEV-4B", "kev-9b": "KEV-9B"}

# Row order follows this dict.
REGIME_LABELS = {
    BASE_SLUG: "Off-the-shelf",
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}

# Column order follows this dict. Keys are the dataset prefixes in ctx.regimes_by_model columns.
DATASET_LABELS = {
    "kriner": "Flights",
    "kantorowicz": "Consumer rights",
    "magistro": "AI & offshoring",
    "grummon": "Food labels",
    "covid": "Vaccines",
}

# Sub-columns under each dataset. Keys are the metric suffixes in ctx.regimes_by_model columns ("kl" holds empirical CE).
METRIC_LABELS = {"mse": "MSE", "kl": "CE"}
METRIC_SCALE = {"mse": 1000.0, "kl": 1.0}
METRIC_DECIMALS = {"mse": 2, "kl": 3}

STUB_HEADERS = ("Model", "Trained on")

## Build the cells

In [4]:
# Bold marks the lowest value per model, dataset, and metric.
rows = []
for model_slug in MODELS:
    regimes_df = ctx.regimes_by_model[model_slug]
    cell_text = {}
    for dataset_key in DATASET_LABELS:
        for metric_key in METRIC_LABELS:
            column = f"{dataset_key}_{metric_key}"
            values = regimes_df.loc[list(REGIME_LABELS), column].astype(float)
            if values.isna().any():
                raise ValueError(f"{model_slug}: missing {column} for {list(values[values.isna()].index)}")
            best_slug = values.idxmin()
            for regime_slug in REGIME_LABELS:
                scaled = float(values.loc[regime_slug]) * METRIC_SCALE[metric_key]
                text = f"{scaled:.{METRIC_DECIMALS[metric_key]}f}"
                if regime_slug == best_slug:
                    text = f"**{text}**"
                cell_text[(regime_slug, dataset_key, metric_key)] = text

    for regime_index, (regime_slug, regime_label) in enumerate(REGIME_LABELS.items()):
        model_cell = MODEL_LABELS[model_slug] if regime_index == 0 else ""
        row = [model_cell, regime_label]
        for dataset_key in DATASET_LABELS:
            for metric_key in METRIC_LABELS:
                row.append(cell_text[(regime_slug, dataset_key, metric_key)])
        rows.append(row)

sub_headers = list(STUB_HEADERS)
for dataset_key in DATASET_LABELS:
    for metric_key in METRIC_LABELS:
        sub_headers.append(METRIC_LABELS[metric_key])
display(pd.DataFrame(rows, columns=sub_headers))

,Model,Trained on,MSE,CE,MSE,CE,MSE,CE,MSE,CE,MSE,CE
0,KEV-0.8B,Off-the-shelf,17.75,1.086,8.34,1.418,33.92,2.283,5.46,1.059,12.07,1.185
1,,Electricity,8.91,0.693,7.35,0.821,29.60,1.689,3.53,0.915,3.73,1.082
2,,Drones,9.68,0.709,7.17,0.806,34.41,1.630,3.25,0.953,4.08,1.105
3,,Drones + Electricity,**4.62**,**0.647**,**6.04**,**0.695**,**26.61**,**1.558**,**2.30**,**0.879**,**2.56**,**1.067**
4,KEV-4B,Off-the-shelf,4.10,0.811,4.89,0.779,99.01,2.434,21.10,1.100,9.08,1.525
5,,Electricity,1.72,0.625,3.27,0.650,113.26,1.886,3.57,0.887,1.73,1.120
6,,Drones,1.00,0.623,2.64,0.663,74.69,1.797,7.04,0.959,4.63,1.195
7,,Drones + Electricity,**0.73**,**0.588**,**2.25**,**0.629**,**63.98**,**1.636**,**2.28**,**0.854**,**1.12**,**1.046**
8,KEV-9B,Off-the-shelf,5.12,0.833,1.76,0.804,82.89,2.437,22.73,1.238,14.30,1.422
9,,Electricity,2.17,0.647,**1.34**,0.694,95.68,1.873,18.85,0.972,4.25,1.203


## Format as a Pandoc grid table

In [5]:
# Pandoc grid table: dataset names span their metric sub-columns in the first header row.
n_stubs = len(STUB_HEADERS)
n_metrics = len(METRIC_LABELS)
n_columns = len(sub_headers)

widths = []
for column_index in range(n_columns):
    widest = len(sub_headers[column_index])
    for row in rows:
        widest = max(widest, len(row[column_index]))
    widths.append(widest)


def span_columns(dataset_index):
    first = n_stubs + n_metrics * dataset_index
    return list(range(first, first + n_metrics))


def span_inner_width(dataset_index):
    columns = span_columns(dataset_index)
    return sum(widths[c] for c in columns) + 3 * (len(columns) - 1)


# Widen a dataset's sub-columns when its name is wider than them.
for dataset_index, dataset_label in enumerate(DATASET_LABELS.values()):
    extra = len(dataset_label) - span_inner_width(dataset_index)
    if extra > 0:
        columns = span_columns(dataset_index)
        for c in columns:
            widths[c] += extra // len(columns)
        widths[columns[-1]] += extra % len(columns)


def padded(text, width):
    return " " + text.ljust(width) + " "


top_parts = [("-" * (widths[c] + 2)) for c in range(n_stubs)]
for dataset_index in range(len(DATASET_LABELS)):
    top_parts.append("-" * (span_inner_width(dataset_index) + 2))
top_border = "+" + "+".join(top_parts) + "+"

header_1_parts = [padded(STUB_HEADERS[c], widths[c]) for c in range(n_stubs)]
for dataset_index, dataset_label in enumerate(DATASET_LABELS.values()):
    header_1_parts.append(padded(dataset_label, span_inner_width(dataset_index)))
header_1 = "|" + "|".join(header_1_parts) + "|"

stub_blanks = [" " * (widths[c] + 2) for c in range(n_stubs)]
metric_rules = ["-" * (widths[c] + 2) for c in range(n_stubs, n_columns)]
header_middle = "|" + "|".join(stub_blanks) + "+" + "+".join(metric_rules) + "+"

metric_cells = [padded(sub_headers[c], widths[c]) for c in range(n_stubs, n_columns)]
header_2 = "|" + "|".join(stub_blanks + metric_cells) + "|"

header_end_parts = []
for c in range(n_columns):
    if c < n_stubs:
        header_end_parts.append(":" + "=" * (widths[c] + 1))
    else:
        header_end_parts.append("=" * (widths[c] + 1) + ":")
header_end = "+" + "+".join(header_end_parts) + "+"

row_rule = "+" + "+".join("-" * (widths[c] + 2) for c in range(n_columns)) + "+"

lines = [top_border, header_1, header_middle, header_2, header_end]
for row in rows:
    lines.append("|" + "|".join(padded(row[c], widths[c]) for c in range(n_columns)) + "|")
    lines.append(row_rule)
table_md = "\n".join(lines) + "\n"

## Save `tbl-results-mse.md` (included by the manuscript)

In [6]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

+----------+----------------------+----------------------+----------------------+-----------------------+-----------------------+----------------------+
| Model    | Trained on           | Flights              | Consumer rights      | AI & offshoring       | Food labels           | Vaccines             |
|          |                      +----------+-----------+----------+-----------+-----------+-----------+-----------+-----------+----------+-----------+
|          |                      | MSE      | CE        | MSE      | CE        | MSE       | CE        | MSE       | CE        | MSE      | CE        |
+:=========+:=====================+=========:+==========:+=========:+==========:+==========:+==========:+==========:+==========:+=========:+==========:+
| KEV-0.8B | Off-the-shelf        | 17.75    | 1.086     | 8.34     | 1.418     | 33.92     | 2.283     | 5.46      | 1.059     | 12.07    | 1.185     |
+----------+----------------------+----------+-----------+----------+-----------+-